# DeepSHAP Gene-Marker Discovery — Approach 3 Classifier × Approach 2 Decoder

> **Goal:** Turn the frozen Approach 3 classifier from `cell_annotation_DL_v2_3.ipynb` into a
> **cell-type × gene marker table**, using DeepSHAP to attribute predictions to the 30-dim latent
> space, then a frozen NB-VAE decoder to translate that latent attribution into gene-level scores.

Based on `deepsharp_genemarker.py` (Book 3, Chapter 4 → §5.4 DeepSHAP → §5.6 marker table).

## Nothing is retrained here
- **scVI / NB-VAE is not re-run** — this notebook loads the frozen **Approach 2** NB-VAE decoder
  weights already saved by `cell_annotation_DL_v2_3.ipynb` (Cell 9).
- **Approach 3 is not re-run** — this notebook loads the frozen `CellTypeNet` classifier weights and
  its predicted labels, already saved by `cell_annotation_DL_v2_3.ipynb` (Cell 11).
- The only **new** computation is DeepSHAP + one Jacobian pass through the frozen decoder + one matrix multiply.

## Inputs (already produced by `cell_annotation_DL_v2_3.ipynb`)

| Input | Source | File |
|---|---|---|
| Latent codes `Z` (`X_emb`, 30-dim) | Approach 3 (Cell 11) | `3.customNN_onXemb/approach3_adata_reannotated.h5ad` |
| Predicted cell-type labels | Approach 3 (Cell 11) | same file, `obs["approach3_celltype"]` |
| Frozen classifier (`CellTypeNet`) | Approach 3 (Cell 11) | `3.customNN_onXemb/hemibrain_nn_model.pt` |
| Frozen NB-VAE decoder | Approach 2 (Cell 9) | `2.customNB-VAE_scANVI_style/hemibrain_scanvi_nbvae_model.pt` |
| HVG gene names (decoder's gene panel) | scdownstream pipeline | `6.integrate/6.1_hvgs/merged_hvgs.h5ad` |

## Output
- `approach3_deepshap_marker_table.json` — top marker genes per predicted cell type.

In [ ]:
from pathlib import Path
import sys

# Run Jupyter with this project folder as its working directory.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "scripts" / "project_paths.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "scripts" / "project_paths.py").is_file():
    raise RuntimeError("Set the notebook working directory to the project folder.")
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from project_paths import atlas_path, data_path


## Step 0 — Setup

In [ ]:
# Cell 1 | code
import os
import json
import numpy as np
import pandas as pd
import scanpy as sc
import torch
import torch.nn as nn
import shap

# ── Same torch compatibility patch used in cell_annotation_DL_v2_3.ipynb ─────
torch.compile = lambda x: x  # No-op: avoids loading torch._inductor
try:
    from torch import _utils
except (ImportError, AttributeError):
    import sys, importlib
    if 'torch' in sys.modules:
        try:
            importlib.reload(sys.modules['torch'])
        except Exception:
            pass
    try:
        from torch import _utils
    except (ImportError, AttributeError):
        class _UtilsStub:
            pass
        sys.modules['torch._utils'] = _UtilsStub()
        torch._utils = _UtilsStub()

torch.manual_seed(0)
np.random.seed(0)

# ── Artifacts already produced by cell_annotation_DL_v2_3.ipynb ──────────────
APPROACH3_DIR = str(data_path('cell_annotation_optimize/3.customNN_onXemb'))
APPROACH2_DIR = str(data_path('cell_annotation_optimize/2.customNB-VAE_scANVI_style'))
HVG_H5AD = str(data_path('scdownstream/6.integrate/6.1_hvgs/merged_hvgs.h5ad'))

OUTPUT_DIR = str(data_path('cell_annotation_optimize/4.deepshap_genemarkers'))
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Approach 3 dir : {APPROACH3_DIR}")
print(f"Approach 2 dir : {APPROACH2_DIR}")
print(f"Output dir     : {OUTPUT_DIR}")

## Step 1 — Load the frozen Approach 3 classifier + its latent input (no retraining)

`Z` is the same 30-dim `X_emb` that Approach 3 classified on. `pred` is Approach 3's own predicted
labels, re-encoded to the same integer class order the `CellTypeNet` was trained with
(`label_classes`, saved alongside the model). The architecture below matches Cell 11 exactly
(including the `Dropout` layers) so the saved `state_dict` keys line up correctly.

In [ ]:
# Cell 2 | code
# ── Step 1a/1b: latent codes (X_emb) + predicted labels, already produced by Approach 3 ──
adata3 = sc.read_h5ad(os.path.join(APPROACH3_DIR, "approach3_adata_reannotated.h5ad"))
Z = adata3.obsm["X_emb"].astype(np.float32)
pred_labels_str = adata3.obs["approach3_celltype"].astype(str).values
n_cells, n_latent = Z.shape
print(f"Latent Z (X_emb): {Z.shape}  from {adata3.n_obs} cells")

# ── Step 1c: frozen Approach 3 classifier (CellTypeNet), loaded NOT retrained ──
nn_ckpt = torch.load(os.path.join(APPROACH3_DIR, "hemibrain_nn_model.pt"), map_location="cpu", weights_only=False)
label_classes = nn_ckpt["label_classes"]
label_to_idx = {c: i for i, c in enumerate(label_classes)}
pred = np.array([label_to_idx[s] for s in pred_labels_str], dtype=np.int64)
n_classes = nn_ckpt["n_classes"]
print(f"Cell types (C={n_classes}): {label_classes[:5]} ...")

class CellTypeNet(nn.Module):
    def __init__(self, n_input, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_input, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128,  64),     nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, n_classes)
        )
    def forward(self, x):
        return self.net(x)   # raw logits (trained with CrossEntropyLoss)

net = CellTypeNet(n_input=nn_ckpt["n_input"], n_classes=n_classes)
net.load_state_dict(nn_ckpt["model_state"])
net.eval()
for p in net.parameters():
    p.requires_grad_(False)

Zt = torch.tensor(Z)
print("Frozen Approach 3 classifier loaded (no retraining).")

## Step 2 — DeepSHAP on the frozen classifier (book §5.4)

For each cell, DeepSHAP attributes the classifier's predicted-class probability to each of the 30
latent dimensions, relative to a background set of cells. `phi` (n_cells × 30) holds each cell's
attribution vector for **its own predicted class**.

The efficiency check confirms `sum(phi) ≈ P(predicted class) − P(predicted class | background)` —
the core SHAP guarantee. A near-zero gap confirms the attribution was computed correctly.

In [ ]:
# Cell 3 | code
# ── Step 2: DeepSHAP on the frozen classifier (book §5.4) ────────────────────
N_EXPLAIN = 9000       # subsample size (of n_cells total)
N_BACKGROUND = 100     # SHAP reference set size
MIN_PER_GROUP = 20     # floor so thin (sample, cell type) combos aren't dropped

rng = np.random.default_rng(0)
bg_idx = rng.choice(n_cells, size=min(N_BACKGROUND, n_cells), replace=False)
background = Zt[bg_idx]

# Stratify the explained subsample by (sample, predicted cell type) so all 3
# samples stay proportionally represented, not just the pooled cell types.
sample_ids = adata3.obs["sample"].astype(str).values
strata = pd.DataFrame({"sample": sample_ids, "pred": pred})
group_sizes = strata.groupby(["sample", "pred"]).size()

if N_EXPLAIN is not None and N_EXPLAIN < n_cells:
    alloc = (group_sizes / n_cells * N_EXPLAIN).round().astype(int)
    alloc = np.maximum(alloc, np.minimum(MIN_PER_GROUP, group_sizes))
    alloc = np.minimum(alloc, group_sizes)

    explain_idx = []
    for (smpl, cls), n_take in alloc.items():
        if n_take <= 0:
            continue
        idx_in_group = np.where((sample_ids == smpl) & (pred == cls))[0]
        explain_idx.append(rng.choice(idx_in_group, size=n_take, replace=False))
    explain_idx = np.concatenate(explain_idx)
    print(f"Stratified subsample: {len(explain_idx)} / {n_cells} cells "
          f"across {len(alloc)} (sample x cell type) groups")
else:
    explain_idx = np.arange(n_cells)

Zt_explain = Zt[explain_idx]
pred_explain = pred[explain_idx]

explainer = shap.DeepExplainer(net, background)
raw = explainer.shap_values(Zt_explain)
if isinstance(raw, list):
    # older shap API: list of C arrays, each (n_explain, n_latent)
    shap_all = np.stack(raw, axis=0)                              # (C, n_explain, n_latent)
    phi = shap_all[pred_explain, np.arange(len(explain_idx)), :]
else:
    # newer shap API: single array (n_explain, n_latent, C)
    shap_all = raw
    phi = np.take_along_axis(shap_all, pred_explain[:, None, None], axis=2)[:, :, 0]

# Efficiency check (§5.4): sum of the 30 phi values ≈ P(pred) - P(baseline)
with torch.no_grad():
    p_cells = torch.softmax(net(Zt_explain), dim=1).numpy()
    p_base = torch.softmax(net(background), dim=1).numpy().mean(0)
gap = p_cells[np.arange(len(explain_idx)), pred_explain] - p_base[pred_explain]
print("mean |sum(phi) - gap| =", round(float(np.abs(phi.sum(1) - gap).mean()), 5),
      " (~0 confirms efficiency)")

In [ ]:
# Cell 3.x | code (cache helpers for DeepSHAP outputs from Cell 3)
DEEPSHAP_CACHE_PATH = os.path.join(OUTPUT_DIR, "deepshap_step2_cache.npz")

# Toggle behavior:
SAVE_CURRENT = True   # Save currently loaded in-memory outputs from Cell 3
LOAD_CACHE = False    # Load cache back into memory (for future sessions)

if SAVE_CURRENT:
    required = ["phi", "explain_idx", "pred_explain", "bg_idx", "gap"]
    missing = [name for name in required if name not in globals()]
    if missing:
        print("Cannot save cache; missing variables from Cell 3:", missing)
    else:
        np.savez_compressed(
            DEEPSHAP_CACHE_PATH,
            phi=phi,
            explain_idx=explain_idx,
            pred_explain=pred_explain,
            bg_idx=bg_idx,
            gap=gap,
        )
        print(f"Saved DeepSHAP cache to: {DEEPSHAP_CACHE_PATH}")

if LOAD_CACHE:
    if os.path.exists(DEEPSHAP_CACHE_PATH):
        cache = np.load(DEEPSHAP_CACHE_PATH, allow_pickle=False)
        phi = cache["phi"]
        explain_idx = cache["explain_idx"]
        pred_explain = cache["pred_explain"]
        bg_idx = cache["bg_idx"]
        gap = cache["gap"]
        print(f"Loaded DeepSHAP cache from: {DEEPSHAP_CACHE_PATH}")
        print("phi shape:", phi.shape, "| explained cells:", len(explain_idx))
    else:
        print(f"Cache file not found: {DEEPSHAP_CACHE_PATH}")

## Step 3 — Aggregate SHAP importance by predicted cell type (book §5.6 step 2)

Average `|phi|` over all cells assigned to each predicted type → `Imp` (C × 30): how much each latent
dimension matters, on average, for predicting each cell type.

In [ ]:
# Cell 4 | code
# ── Step 3: aggregate SHAP importance by predicted cell type (book §5.6 step 2) ──
Imp = np.zeros((n_classes, n_latent))
for k in range(n_classes):
    rows = phi[pred_explain == k]
    if len(rows):
        Imp[k] = np.abs(rows).mean(0)

n_with_cells = (Imp.sum(axis=1) > 0).sum()
print(f"Per-type latent importance Imp: {Imp.shape}  ({n_with_cells}/{n_classes} types represented)")

## Step 4 — Decode latent importance into gene-level loadings (book §5.6 step 3)

To turn "dimension 7 matters for type X" into "gene ABC1 matters for type X", we need a **decoder**
mapping the 30-dim latent back to gene space, and take its local Jacobian (`L`, 30 × n_HVG) around a
representative latent point.

**Which decoder?** The only fully self-contained NB decoder in this project is **Approach 2's own
NB-VAE decoder** (Cell 9) — its exact architecture and weights are fully known, with no external
dependencies. We use it here as the decoder for Approach 3's latent attributions.

> ⚠️ **Approximation note:** Approach 3's `X_emb` and Approach 2's `X_scANVI` are two different latent
> embeddings of the same cells (from two independently-trained encoders). Using Approach 2's decoder
> to interpret Approach 3's latent directions is therefore a **practical proxy**, not an exact
> generative decode — both encode the same underlying HVG expression, so the resulting gene ranking is
> a reasonable approximation, but not mathematically exact. For an exact decode you would need the
> decoder that actually produced `X_emb` (the scdownstream pipeline's
> `6.2.1_NBVAE/merged_nbvae_model.pt`), which requires that pipeline's own covariate encoding — not
> reconstructable from this workspace alone.

In [ ]:
# Cell 5 | code
# ── Step 4: decoder loadings L = d(log gene-scale)/dz, from Approach 2's frozen NB-VAE decoder ──
# Architecture copied verbatim from Cell 9 (Approach 2) so the saved state_dict keys match.
class Decoder(nn.Module):
    def __init__(self, n_latent, n_hidden, n_output):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_latent, n_hidden[1]),
            nn.BatchNorm1d(n_hidden[1]),
            nn.ReLU(),
            nn.Linear(n_hidden[1], n_hidden[0]),
            nn.BatchNorm1d(n_hidden[0]),
            nn.ReLU(),
        )
        self.fc_scale = nn.Linear(n_hidden[0], n_output)
        self.log_theta = nn.Parameter(torch.zeros(n_output))

    def forward(self, z, log_library):
        h = self.net(z)
        px_scale = torch.softmax(self.fc_scale(h), dim=-1)
        px_mu = px_scale * torch.exp(log_library).unsqueeze(1)
        px_theta = torch.exp(self.log_theta)
        return px_mu, px_theta

vae_ckpt = torch.load(os.path.join(APPROACH2_DIR, "hemibrain_scanvi_nbvae_model.pt"), map_location="cpu", weights_only=False)
vae_params = vae_ckpt["params"]

decoder_state = {k[len("decoder."):]: v for k, v in vae_ckpt["model_state"].items() if k.startswith("decoder.")}
n_hvg = decoder_state["fc_scale.weight"].shape[0]

decoder = Decoder(n_latent=vae_params["n_latent"], n_hidden=vae_params["n_hidden"], n_output=n_hvg)
decoder.load_state_dict(decoder_state)
decoder.eval()
for p in decoder.parameters():
    p.requires_grad_(False)

# Gene names: the HVG panel Approach 2's decoder was trained to reconstruct.
adata_hvg = sc.read_h5ad(HVG_H5AD)
genes = adata_hvg.var_names.to_numpy()
assert len(genes) == n_hvg, f"gene count mismatch: {len(genes)} vs decoder output {n_hvg}"

def decode_log_scale(z):
    # log of the *relative* gene-expression scale (px_scale), independent of library size —
    # matches the original script's use of px_scale rather than the library-scaled px_mu.
    h = decoder.net(z)
    scale = torch.softmax(decoder.fc_scale(h), dim=-1)
    return torch.log(scale + 1e-8)

z0 = background.mean(dim=0, keepdim=True).requires_grad_(True)
J = torch.autograd.functional.jacobian(decode_log_scale, z0)
L = J.squeeze().T.detach().numpy()          # (n_latent x n_HVG)
print("Decoder loading matrix L:", L.shape, " (n_latent x n_HVG)")

## Step 5 — Build the cell-type × gene marker table (book §5.6 steps 4–5)

`M = Imp @ |L|` (C × n_HVG): each row is a cell type's gene-importance vector. Take the top-N genes per
row as that type's marker panel.

In [ ]:
# Cell 6 | code
# ── Step 5: marker table  M = Imp . |L|  (book §5.6 steps 4-5) ───────────────
M = Imp @ np.abs(L)                          # (C x n_HVG)

top_n = 15
table = {}
for k in range(n_classes):
    if Imp[k].sum() == 0:
        continue   # no cells predicted this type in the explained subset
    order = np.argsort(-M[k])[:top_n]
    table[label_classes[k]] = [(str(genes[j]), float(round(M[k][j], 4))) for j in order]

for t, marker_list in table.items():
    print(f"{t}:  " + ", ".join(g for g, _ in marker_list[:8]))

marker_path = os.path.join(OUTPUT_DIR, "approach3_deepshap_marker_table.json")
with open(marker_path, "w") as f:
    json.dump(table, f, indent=2)
print(f"\nSaved marker table: {marker_path}")

## Notes

- **Runtime:** DeepSHAP over all ~26k cells can take a few minutes on CPU. Set `N_EXPLAIN` in Step 2
  to a smaller number (e.g. `5000`) for a quick pass while iterating.
- **Gene IDs:** `genes` are the Ensembl IDs from the HVG panel (`merged_hvgs.h5ad.var_names`) — map to
  gene symbols separately if needed for reporting.
- **Alternative, fully self-consistent pipeline:** instead of pairing Approach 3's classifier with
  Approach 2's decoder (an approximation — see Step 4's caveat), you could instead run DeepSHAP on
  **Approach 2's own classifier head** (`SemiSupervisedNBVAE.classifier`) using `X_scANVI` as `Z` and
  `approach2_celltype` as the predicted labels. That pairing has zero latent-space mismatch since both
  come from the same trained model, at the cost of using Approach 2's (currently NB-VAE) predictions
  instead of Approach 3's.